# 02 — Train/test split and baseline

Goal: split the ratings into a training set and a test set, then measure the simplest possible model.

In [1]:
import sys
sys.path.append("../src")

from recommender.data import load_ratings, load_movies

ratings = load_ratings()
movies = load_movies()

print("Ratings loaded:", len(ratings))

Ratings loaded: 100000


In [2]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42)

print("Training ratings:", len(train))
print("Test ratings:    ", len(test))
print("Average rating in train:", round(train["rating"].mean(), 4))
print("Average rating in test: ", round(test["rating"].mean(), 4))

train.head(3)

Training ratings: 80000
Test ratings:     20000
Average rating in train: 3.5313
Average rating in test:  3.5242


,user_id,movie_id,rating,timestamp
75220,807,1411,1,1998-04-20 14:30:19
48955,474,659,5,1998-02-19 21:53:07
44966,463,268,4,1997-10-20 22:02:20


In [5]:
import numpy as np

global_mean = train["rating"].mean()

errors = test["rating"] - global_mean

mae = errors.abs().mean()
rmse = np.sqrt((errors ** 2).mean())

print("The lazy guesser always answers:", round(global_mean, 4))
print("MAE: ", round(mae, 4))
print("RMSE:", round(rmse, 4))

The lazy guesser always answers: 3.5313
MAE:  0.942
RMSE: 1.1239


In [6]:
movie_means = train.groupby("movie_id")["rating"].mean()

predictions = test["movie_id"].map(movie_means)

print("Test ratings with no guess:", predictions.isna().sum())

predictions = predictions.fillna(global_mean)

errors = test["rating"] - predictions
mae = errors.abs().mean()
rmse = np.sqrt((errors ** 2).mean())

print("Star Wars average in train:", round(movie_means.loc[50], 2))
print("MAE: ", round(mae, 4))
print("RMSE:", round(rmse, 4))

Test ratings with no guess: 31
Star Wars average in train: 4.37
MAE:  0.8123
RMSE: 1.021


In [7]:
stats = train.groupby("movie_id")["rating"].agg(mean_rating="mean", n_ratings="count")
stats = movies[["movie_id", "title"]].merge(stats, on="movie_id")

stats.sort_values("mean_rating", ascending=False).head(5).round(2)

,movie_id,title,mean_rating,n_ratings
1290,1293,Star Kid (1997),5.0,2
1490,1500,Santa with Muscles (1996),5.0,2
1620,1642,Some Mother's Son (1996),5.0,1
1303,1306,Delta of Venus (1994),5.0,1
1186,1189,Prefontaine (1997),5.0,2


In [ ]:
popular = stats[stats["n_ratings"] >= 50].sort_values("mean_rating", ascending=False)

print("Movies with at least 50 ratings:", len(popular))
popular.head(10).round(2)